<img src="../logo.png" alt="Vegeta" width="240">

# Race design factors — the four courses combined into design requirements (race_design_factors)

Reads the runs the race notebooks 01–04 exported (`races/runs/notebook/<course>/<run>/`); it reruns no race. Three views, kept apart (`races.requirements`):

1. **per course** — measured statistics of each course's valid lap, no margins;
2. **pooled** — the valid laps end to end, every moving second weighted by time (what racing all four courses is like on average);
3. **worst-case envelope** — for each demand the worst valid course (maximum of a demand, minimum of an allowance), times a stated margin: the **requirement**.

**Validity.** Only finished laps count. A DNF (too many collisions, timeout) is listed with its reason and kept out of every number: its peaks are a crash's, not the course's. Elapsed simulated time and wall time are reported separately.

**Margins** (`races.requirements.MARGIN`): speed × 1.1, acceleration and braking × 1.2, lateral g × 1.2, manoeuvre load factor × 1.3, body rates × 1.3, climb and descent rates × 1.2, power × 1.2, energy × 1.25 (and 3 laps for the pack); clearances and turn radius none (they are allowances).

**Formulas.** Percentiles over the moving part of a lap (speed > 1 m/s), weighted by time step. Load factor n = thrust / weight; braking p95 = −p5 of the along-track acceleration; endurance = the reference pack's usable energy / the lap's mean power. **Limitations**: one lap per course and simulator; the mock's body rates are estimates (thrust-vector slew, heading rate); the reference quad flies the line, so the demands are this aircraft's answer to the course, not the course alone; no wind; the hover thrust / weight requirement and an endurance requirement are **unavailable** (no run measures control authority, no race duration is specified).

In [ ]:
# The races are fast and nothing in them is mocked: the laps below are the point-mass mock flyer, and with
# GENERATE_VIDEO = True a real MuJoCo lap (in Vegeta this cell switched vegeta.mock on, which patched nothing here).
DEBUG = False

In [ ]:
PREFER = ("mujoco", "mock")    # per course: the newest valid run of the first simulator in this order
RACE_LAPS = 3

In [ ]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
sys.path.insert(0, "../..")
from races import analysis, courses, export, refquad, requirements as rq
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)
BOOK = {"hangar": "01_hangar", "industrial": "02_industrial", "farm": "03_farm", "quarry": "04_quarry"}
DATA = Path("../data/design_factors"); DATA.mkdir(parents=True, exist_ok=True)   # handed back to Vegeta: designs/data/races/

## 1. The runs on file

In [ ]:
ALL = export.runs()
VALID, FAILED = rq.split(ALL)
display(pd.DataFrame([{"course": r["course"], "simulator": r["provenance"]["simulator"], "status": r["status"], "lap time [s]": r["lap_time_s"],
                       "elapsed sim time [s]": r["elapsed_sim_time_s"], "wall time [s]": r["wall_time_s"], "run": Path(r["dir"]).name} for r in ALL]) if ALL else "no runs")
display(Markdown("**Failed or incomplete runs** (kept out of every number):")); display(FAILED if len(FAILED) else "none")
CHOSEN, MISSING = {}, []
for slug in BOOK:
    pick = None
    for simname in PREFER:
        c = [r for r in VALID if r["course"] == slug and r["provenance"]["simulator"] == simname]
        if c:
            pick = c[-1]; break
    if pick: CHOSEN[slug] = pick
    else: MISSING.append(f"{slug}: no valid lap on file — run notebooks/races/{BOOK[slug]}.ipynb")
print("\n".join(MISSING) or "a valid lap for every course")
assert CHOSEN, "no valid lap on file: run the race notebooks 01–04 first"
MOCK_ONLY = all(r["provenance"]["simulator"] == "mock" for r in CHOSEN.values())
display(pd.DataFrame({s: {"simulator": r["provenance"]["simulator_version"], "run": Path(r["dir"]).name, "lap time [s]": r["lap_time_s"], "collisions": r["collisions"],
                          "course sha256": r["provenance"]["course_sha256"][:12], "model sha256": r["provenance"]["model_sha256"][:12]} for s, r in CHOSEN.items()}))

## 2. Per course (measured, no margins)

In [ ]:
REPORTS = {}
for slug, r in CHOSEN.items():
    lap = export.read_lap(r)
    REPORTS[f"{r['name']} ({r['provenance']['simulator']})"] = analysis.lap_report(lap["history"], courses.COURSES[slug](**r["provenance"]["course_params"]), lap["events"])
PER = rq.per_course(REPORTS)
display(PER.round(2))
analysis.plot_distributions(REPORTS); plt.show()
analysis.plot_corner_envelope(REPORTS); plt.show()

## 3. Pooled (all valid laps end to end, time-weighted)

In [ ]:
POOL = rq.pooled(REPORTS)
display(POOL.round(3).to_frame("pooled"))

## 4. Worst-case design envelope and requirements

In [ ]:
ENV = rq.envelope(PER, mock_only=MOCK_ONLY, race_laps=RACE_LAPS)
display(ENV)
if MOCK_ONLY:
    display(Markdown("**Every lap here is the mock flyer's**: the requirements are this tracker's answer to the courses; the body rates are estimates. A MuJoCo lap (`GENERATE_VIDEO = True` in 01–04) replaces it where it finishes."))
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, key in zip(ax, ("speed max [m/s]", "lateral g p95", "manoeuvre load factor p99 (thrust / weight)", "power p90 [W]")):
    PER.loc[key].astype(float).plot.bar(ax=a, color="#3377dd", title=key)
    req = ENV.loc[key, "requirement"]
    if isinstance(req, float): a.axhline(req, color="#d62828", ls="--", lw=1, label="requirement"); a.legend(fontsize=8)
    a.grid(alpha=0.3, axis="y"); a.set_xlabel("")
plt.tight_layout(); plt.show()
display(pd.Series(refquad.summary(), name="reference quad (what it can do)").to_frame())

## 5. Export

`races/data/design_factors/race_design_factors.json` — handed back to the designers in Vegeta as `designs/data/races/race_design_factors.json` — (the three views, the margins, the runs used and the failed ones) and `race_design_factors_comparison.csv` (per course, pooled column-free, envelope side by side).

In [ ]:
def plain(x):
    if isinstance(x, dict): return {str(k): plain(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [plain(v) for v in x]
    if isinstance(x, (np.floating, float)): return None if not np.isfinite(float(x)) else round(float(x), 5)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.bool_): return bool(x)
    return x
doc = {"schema": 1, "notebook": "races/race_design_factors", "definitions": rq.__doc__, "margins": rq.MARGIN, "race_laps": RACE_LAPS, "mock_only": MOCK_ONLY,
       "runs_used": {s: {"dir": r["dir"], "simulator": r["provenance"]["simulator_version"], "provenance": r["provenance"]} for s, r in CHOSEN.items()},
       "failed_runs": FAILED.to_dict(orient="records") if len(FAILED) else [], "missing": MISSING,
       "per_course": PER.to_dict(), "pooled": POOL.to_dict(), "envelope": ENV.to_dict(orient="index")}
(DATA / "race_design_factors.json").write_text(json.dumps(plain(doc), indent=1, ensure_ascii=False))
CMP = PER.copy()
CMP["worst case"] = ENV["worst case"]; CMP["margin"] = ENV["margin"]; CMP["requirement"] = ENV["requirement"]
CMP.to_csv(DATA / "race_design_factors_comparison.csv")
print("exported", DATA / "race_design_factors.json", "and", DATA / "race_design_factors_comparison.csv")